In [1]:
# ============ TEST SET RUNS — Gemma3 translation (FR + DE) ============
import os, re, json, requests
from datetime import datetime
from tqdm import tqdm
import torch
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset, Dataset

def find_cache_dir():
    for c in ['baseline/cache', 'cache', '../baseline/cache']:
        if os.path.exists(c): return c
CACHE_DIR = find_cache_dir()

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
batch_size = 16
TOP_K = 1000
RUN_NAME = "TUWien-Track3-gemma3"
OLLAMA = "http://localhost:11434/api/generate"
MT_MODEL = "gemma3:1b"
print(f"Device: {device}")

collection_embeddings = torch.load(os.path.join(CACHE_DIR, 'collection_embeddings.pt'))
collection_keys = np.load(os.path.join(CACHE_DIR, 'collection_keys.npy'))

e5_name = 'intfloat/multilingual-e5-large'
tok = AutoTokenizer.from_pretrained(e5_name)
model = AutoModel.from_pretrained(e5_name).to(device).eval()
print("E5 loaded.")

def avg_pool(h, m):
    h = h.masked_fill(~m[..., None].bool(), 0.0)
    return h.sum(dim=1) / m.sum(dim=1)[..., None]

def clean_tweet(t):
    t = re.sub(r'http\S+|www\.\S+', '', t)
    t = re.sub(r'@\w+', '', t)
    t = re.sub(r'#(\w+)', r'\1', t)
    t = re.sub(r'^\d+/n\s*Fact\s*\d+\s*:\s*', '', t, flags=re.IGNORECASE)
    t = re.sub(r'^\d+/\d+\s*', '', t)
    t = re.sub(r'[^\w\s\.\,\:\;\!\?\-\(\)\']+', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def gemma_translate(text, src_lang_name, retries=2):
    prompt = (f"Translate the following {src_lang_name} sentence into English. "
              f"Output ONLY the English translation, no preamble.\n\n"
              f"{src_lang_name}: {text}\n\nEnglish:")
    payload = {"model": MT_MODEL, "prompt": prompt, "stream": False, "keep_alive": "60m",
               "options": {"temperature": 0.0, "num_predict": 200}}
    for attempt in range(retries + 1):
        try:
            r = requests.post(OLLAMA, json=payload, timeout=60)
            r.raise_for_status()
            return r.json()["response"].strip().split("\n")[0]
        except Exception:
            if attempt == retries: return text
            continue

def embed(texts, max_len=160):
    ds = Dataset.from_dict({"text": [f'query: {t}' for t in texts]})
    ds = ds.map(lambda e: tok(e["text"], max_length=max_len, truncation=True, padding="max_length", return_tensors="pt"),
                batched=True, remove_columns=ds.column_names).with_format("torch", columns=["input_ids", "attention_mask"])
    embs = []
    with torch.inference_mode():
        for b in tqdm(DataLoader(ds, batch_size=batch_size, num_workers=0), desc="Embed"):
            b = {k: v.to(device) for k, v in b.items()}
            embs.append(avg_pool(model(**b).last_hidden_state, b['attention_mask']).cpu())
    return F.normalize(torch.vstack(embs), p=2, dim=1)

def write_trec(out_path, indices, embs, run_name):
    cs = (embs @ collection_embeddings.T).numpy()
    top_idxs = np.argsort(-cs, axis=1)[:, :TOP_K]
    top_scores = np.take_along_axis(cs, top_idxs, axis=1)
    top_keys = collection_keys[top_idxs]
    with open(out_path, 'w') as f:
        for i, qid in enumerate(indices):
            for r in range(TOP_K):
                f.write(f"{qid} Q0 {top_keys[i,r]} {r+1} {top_scores[i,r]:.6f} {run_name}\n")
    print(f"Wrote {out_path}")

# Test setleri
test_data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "test")
os.makedirs("runs", exist_ok=True)

LANG_NAMES = {"fr": "French", "de": "German"}

for lang in ["fr", "de"]:
    print(f"\n=== {lang.upper()} TEST GEMMA3 ===")
    split = test_data[lang]
    indices = list(split["index"])
    cleaned = [clean_tweet(t) for t in split["text"]]
    
    print(f"Translating {len(cleaned)} {lang.upper()} test tweets...")
    en_trans = []
    for t in tqdm(cleaned, desc=f"Gemma3 {lang}->en"):
        en_trans.append(gemma_translate(t, LANG_NAMES[lang]))
    
    with open(os.path.join(CACHE_DIR, f'test_en_{lang}_gemma3.json'), 'w', encoding='utf-8') as f:
        json.dump(en_trans, f, ensure_ascii=False)
    
    embs = embed(en_trans)
    write_trec(f"runs/{lang}_test_run_gemma3.txt", indices, embs, RUN_NAME)

# Top-100 versions
for lang in ["fr", "de"]:
    src = f"runs/{lang}_test_run_gemma3.txt"
    dst = f"runs/{lang}_test_run_gemma3_top100.txt"
    with open(src) as fin, open(dst, 'w') as fout:
        prev = None; r = 0
        for line in fin:
            qid = line.split()[0]
            if qid != prev: prev = qid; r = 0
            r += 1
            if r <= 100: fout.write(line)
    sz = os.path.getsize(dst) / 1024 / 1024
    print(f"  {dst}: {sz:.1f} MB")

print("\nDone.")

/Users/ekinkeskin/air2026ss-checkthat-task1-checkthat-task1-methodology-3-group-1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8070.38it/s]


E5 loaded.

=== FR TEST GEMMA3 ===
Translating 1220 FR test tweets...


Embed: 100%|██████████| 77/77 [02:29<00:00,  1.94s/it]


Wrote runs/fr_test_run_gemma3.txt

=== DE TEST GEMMA3 ===
Translating 876 DE test tweets...


Embed: 100%|██████████| 55/55 [01:37<00:00,  1.77s/it]


Wrote runs/de_test_run_gemma3.txt
  runs/fr_test_run_gemma3_top100.txt: 5.2 MB
  runs/de_test_run_gemma3_top100.txt: 3.7 MB

Done.
